# QLoRA fine-tune: Qwen3-4B → crossword-generator SLM

Distills the (Claude + verifier + scaffolding) pipeline into one-shot generation. Trains on `data/sft/train.jsonl` (chat: fixed system contract → minimal size-routed user prompt → verified assistant program), **response-only loss**, dev for validation, `eval` held out for the base-vs-tuned test (see `colab_eval.ipynb`).

## 1. Install (Qwen3 needs recent transformers; the Llama-2-era pins won't load it)

In [ ]:
!pip install -q -U 'transformers>=4.51.0' 'trl>=0.12.0' 'peft>=0.13.0' 'bitsandbytes>=0.44.0' 'accelerate>=1.0.0' 'datasets>=3.0.0'

## 2. Get the data
Either clone the repo and (re)build the merged, upsampled splits, **or** upload `train.jsonl`/`dev.jsonl`/`eval.jsonl` yourself and set `DATA_DIR` to their folder.

In [ ]:
REPO_URL = "https://github.com/Avaneesh-Ramesh-07/<REPO>.git"
import os
if not os.path.exists("slm"):
    !git clone -q $REPO_URL slm
# rebuild consolidated + upsampled splits (idempotent)
!cd slm && python pipeline/merge_dataset.py --upsample 11=3,15=3
DATA_DIR = "slm/data/sft"   # <-- or point at a folder you uploaded
print("data dir:", DATA_DIR, os.listdir(DATA_DIR))

## 3. Config

In [ ]:
# Qwen3-4B instruct. Confirm the exact HF id (alts: "Qwen/Qwen3-4B",
# "Qwen/Qwen3-4B-Instruct"). Start from Instruct for fast SFT.
model_name  = "Qwen/Qwen3-4B-Instruct-2507"  # base model to fine-tune FROM
adapter_dir = "qwen3-4b-crossword-qlora"      # OUTPUT: trained LoRA adapter dir (merged -> adapter_dir + "-merged")
output_dir  = "results"                       # trainer checkpoints + logs

# QLoRA / LoRA
lora_r, lora_alpha, lora_dropout = 32, 64, 0.05
# Qwen attention + MLP projections
target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

# programs are long (a full generator); give the sequence room
max_seq_length = 4096

num_train_epochs = 3
per_device_train_batch_size = 1
per_device_eval_batch_size  = 1
gradient_accumulation_steps = 16   # effective batch ~16
learning_rate = 2e-4
lr_scheduler_type = "cosine"
warmup_ratio = 0.03
weight_decay = 0.0
logging_steps = 10

## 4. Load Qwen3-4B in 4-bit (QLoRA) + LoRA adapters

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name, quantization_config=bnb_config, device_map={"": 0},
    torch_dtype=torch.bfloat16, trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

peft_config = LoraConfig(
    r=lora_r, lora_alpha=lora_alpha, lora_dropout=lora_dropout,
    target_modules=target_modules, bias="none", task_type="CAUSAL_LM",
)

## 5. Load data + render the Qwen chat template
Each row is `{messages:[system,user,assistant]}`. We render it with the model's own chat template into a `text` field; the response-only collator (next cell) then masks everything up to the assistant turn so loss is computed **only on the program**.

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files={
    "train": f"{DATA_DIR}/train.jsonl",
    "dev":   f"{DATA_DIR}/dev.jsonl",
}, )

def render(row):
    # add_generation_prompt=False -> include the assistant turn as the target
    return {"text": tokenizer.apply_chat_template(row["messages"], tokenize=False,
                                                   add_generation_prompt=False)}

ds = ds.map(render, remove_columns=[c for c in ds["train"].column_names if c != "text"])
print(ds)
print("\n--- one rendered example (head) ---\n", ds["train"][0]["text"][:600])

## 6. Response-only loss
Qwen renders the assistant turn after `<|im_start|>assistant\n`. Masking up to that marker means gradients flow only through the generated program, not the (fixed) system contract or user prompt.

In [ ]:
from trl import DataCollatorForCompletionOnlyLM
response_template = "<|im_start|>assistant\n"   # Qwen chat-template assistant marker
collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)
# sanity: confirm the marker tokenizes and is found in a sample
assert response_template in ds["train"][0]["text"], "assistant marker not found — check template"

## 7. Train (dev = in-training validation; eval stays untouched)

In [ ]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=learning_rate,
    lr_scheduler_type=lr_scheduler_type,
    warmup_ratio=warmup_ratio,
    weight_decay=weight_decay,
    logging_steps=logging_steps,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_seq_length=max_seq_length,
    dataset_text_field="text",
    packing=False,   # required for response-only masking
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="tensorboard",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["dev"],
    peft_config=peft_config,
    data_collator=collator,
)
trainer.train()

## 8. Save LoRA adapter + merged fp16 model

In [ ]:
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print("saved LoRA adapter to", adapter_dir)

# merge to a standalone fp16 model for inference / GGUF export
from peft import PeftModel
import torch, gc
del model, trainer; gc.collect(); torch.cuda.empty_cache()
base = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16,
                                            device_map={"": 0}, trust_remote_code=True)
merged = PeftModel.from_pretrained(base, adapter_dir).merge_and_unload()
merged.save_pretrained(adapter_dir + "-merged")
tokenizer.save_pretrained(adapter_dir + "-merged")
print("saved merged model to", adapter_dir + "-merged")

## 9. Persist to Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/slm_ckpt
# {adapter_dir} is interpolated by IPython from the Python namespace at runtime
!cp -r {adapter_dir} {adapter_dir}-merged /content/drive/MyDrive/slm_ckpt/ 2>/dev/null; echo saved

## Next
Run **`colab_eval.ipynb`** to serve this merged model and score it on the held-out `eval.jsonl` through our sandbox+scorer — the tuned side of the base-vs-tuned table in `GAP_ANALYSIS.md` (unaugmented Opus is ~5–7% valid; target is high pass@1).